#Collection, Loading and Analysis of FOMC Dataset.

## Step One: Importing Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

## Step two : Install datasets and loading Dataset from huggingface

In [ ]:
pip install datasets

In [ ]:
from datasets import load_dataset

In [ ]:
dataset = load_dataset("vtasca/fomc-statements-minutes") ## Loading FOMC dataset

In [ ]:
print(dataset)

## Step three : FOMC Dataset Analysis

In [ ]:
## Converting to Python dataframe for processing.

fomc_df = dataset["train"].to_pandas()

In [ ]:
##Checking first few records

fomc_df.head()

In [ ]:
fomc_df.shape ## Number of rows and columns check

### There are four collumns in dataset and total 458 rows/records and 5 columns.

In [ ]:
## Check for missing values
fomc_df.isnull().sum() ## ## RElease date have 29 null values

In [ ]:
## Check inofmration of FOMC columns
fomc_df.info()

Release Date column have null values

In [ ]:
fomc_df['Release Date'].describe()

### Record spans from Feb 2000 to Dec 2025 with latest record of 10 Dec 2025.

In [ ]:
##Checking sample text record of Minute
fomc_df['Text'][0]

In [ ]:
##Checking sample text record of Statement
fomc_df['Text'][1]

### Statement is less lengthy as compared to  minutes of FOMC.

In [ ]:
print('Count of Minutes -',fomc_df[fomc_df['Type']=='Minute'].shape[0])
print('Count of Statements -',fomc_df[fomc_df['Type']=='Statement'].shape[0])

There are 238 Minutes and 220 Statements in dataset.

### A] EDA of FOMC dataset

In [ ]:
## Converting object types to datetime of Release Date and Release Year column as they contain date values.

fomc_df['Release Date'] = pd.to_datetime(fomc_df['Release Date'])
fomc_df['Date'] = pd.to_datetime(fomc_df['Date'])

In [ ]:
## To analyse days difference between Release date and actul date of communication.

fomc_df['Date Difference'] = (fomc_df['Release Date'] - fomc_df['Date']).dt.days
fomc_df['Date Difference']=fomc_df['Date Difference'].round().astype('Int64')
fomc_df['Date Difference'].value_counts().sort_index(ascending=False)

Maximum difference in days between actual communication date and release date of Minute is 60 days.
<br>However, 149 minutes have difference days of 21 days.

In [ ]:
fomc_df[fomc_df['Type']=='Statement']['Date Difference'].value_counts() ## Analysing date differences of Statement

In [ ]:
fomc_df[fomc_df['Type']=='Minute']['Date Difference'].value_counts() ## Analysing date differences of Minute

Statements are release on same date when meeting is conducted, however Minutes are released 21 days post the actual communication date.

In [ ]:
## Adding a column of Release year to get year from date of release.

fomc_df['Release Year']=fomc_df['Release Date'].dt.year
fomc_df['Release Year']=fomc_df['Release Year'].round().astype('Int64')

In [ ]:
fomc_df['Release Year'].value_counts() ## Can see release frequency were highest in 2007.

In [ ]:
## Amending the Release date of Statement record having Date difference as '7' to rectify outlier.

fomc_df[(fomc_df['Date Difference']==7)& (fomc_df['Type']=='Statement')] ## To get the index location of the record


In [ ]:
fomc_df.loc[104,'Release Date'] = fomc_df.loc[104,'Date']  ## To keep the release date of Statement record same as of actual date.
fomc_df['Date Difference'] = (fomc_df['Release Date'] - fomc_df['Date']).dt.days ## To update the date difference.
fomc_df[(fomc_df['Date Difference']==7)& (fomc_df['Type']=='Statement')] ## To check if successfully updated

In [ ]:
fomc_df['Date Difference']=fomc_df['Date Difference'].round().astype('Int64')
fomc_df[fomc_df['Type']=='Statement']['Date Difference'].value_counts() ## Analysing date differences of Statement


Now we have all statements with date difference as 0.

In [ ]:
## Analysis of records having null values in Release date
fomc_df[fomc_df['Release Date'].isna()]['Type'].value_counts()  ## Can see for records that have 'null' Release Dates are all Minutes.


In [ ]:
## For null values of Release Date of Minutes, updating the value with difference of 21 days from actual date as most of the minutes have difference of 21 days.
ind_list = fomc_df[fomc_df['Release Date'].isna()].index.tolist()  ### Row index to update
for i in ind_list:
    fomc_df.loc[i,'Release Date'] = fomc_df.loc[i,'Date'] + pd.Timedelta(days=21)

fomc_df[fomc_df['Release Date'].isna()] ## To check if any null values present



In [ ]:
fomc_df['Date Difference'] = (fomc_df['Release Date'] - fomc_df['Date']).dt.days ## To update the date difference.
fomc_df['Release Year']=fomc_df['Release Date'].dt.year                          ## To update Release year
fomc_df['Release Year']=fomc_df['Release Year'].round().astype('Int64')


In [ ]:
## Checking if any null values are present

fomc_df.isna().sum()

There are no null values and outliers are handled in FOMC dataset.

In [ ]:
## To save the updated dataset into local excel sheet for future references.
fomc_df.to_excel(r'C:\Users\prapt\Documents\fomc_output.xlsx', index=False)

## Visualisation of dataset

In [ ]:
## Chart of release frequency year-wise.

fomc_df['Release Year'].value_counts().sort_index().plot(kind='bar',color='Slategray',figsize=(10, 6))
plt.title('FOMC Release Frequency Year-wise')
plt.xlabel('Year of Release')
plt.ylabel('Number of Releases')
plt.show()


In [ ]:
fomc_df[fomc_df['Type']=='Minute']['Release Year'].value_counts().sort_index().plot(kind='line', figsize=(8, 4), label='Minute')
fomc_df[fomc_df['Type']=='Statement']['Release Year'].value_counts().sort_index().plot(kind='line', figsize=(8, 4), label='Statement')
plt.title('FOMC Release Frequency Year-wise')
plt.xlabel('Year of Release')
plt.ylabel('Number of Statements/Minutes')
plt.legend()
plt.grid(True)
plt.show()


Findings :

In [ ]:
  ## To plot gaps between release frequency of Statements.

  fomc_df[fomc_df['Type']=='Statement'].sort_values('Release Date')['Release Date'].diff().dt.days.astype('Int64')[1:].sort_values(ascending=False).value_counts()\
                                    .plot(kind='bar',color='olive',figsize=(8,5))
  plt.title('Distribution of Release gaps in days for Statements')
  plt.xlabel('Release gap(days)')
  plt.ylabel('Frequency')
  plt.show()

In [ ]:
  ## To plot gaps between release frequency of Minutes.

  fomc_df[fomc_df['Type']=='Minute'].sort_values('Release Date')['Release Date'].diff().dt.days.astype('Int64')[1:].sort_values(ascending=False).value_counts()\
                                    .plot(kind='bar',color='olive',figsize=(8,5))
  plt.title('Distribution of Release gaps in days for FOMC minutes')
  plt.xlabel('Release gap(days)')
  plt.ylabel('Frequency')
  plt.show()

In [ ]:
## To check average length of Minute and Statements required for analysis.
## Creating new datframe capturing number of words in Minutes and Statements.

fomc_length_df = fomc_df[['Type','Release Date','Release Year']]  ## To copy only required column.


In [ ]:
## To calculate length by words count in document.
for i in fomc_df.index:
  fomc_length_df.loc[i,'Word_Count']=len(fomc_df['Text'][i].split())

In [ ]:
fomc_length_df.head()

In [ ]:
## Visulaisation of word count distribution of Minutes based on word-count

fomc_length_df[fomc_length_df['Type']=='Minute']['Word_Count'].plot(kind='box',figsize=(5,4))
plt.title('Box-plot of distribution of length of Minutes')
plt.ylabel('Count Frequency')
plt.show()
print('\n Below is the percentile distribution of Minute length based on word count :-\n')
fomc_length_df[fomc_length_df['Type']=='Minute']['Word_Count'].describe(percentiles=[0.10, 0.20, 0.50, 0.75, 0.80, 0.9,0.99])


In [ ]:
## Visulaisation of word count distribution of Statements based on word-count

fomc_length_df[fomc_length_df['Type']=='Statement']['Word_Count'].plot(kind='box',figsize=(5,4))
plt.title('Box-plot of distribution of length of Statement')
plt.ylabel('Count Frequency')
plt.show()
print('\n Below is the percentile distribution of Statement length based on word count :-\n')
fomc_length_df[fomc_length_df['Type']=='Statement']['Word_Count'].describe(percentiles=[0.10, 0.20, 0.50, 0.75, 0.80, 0.9,0.99])

In [ ]:
## To check number of sentences in Minutes and Statements.
## Importing NLTK's punkt_tab tokenizer for sentence detection.

import nltk
nltk.download('punkt_tab')

In [ ]:
from nltk.tokenize import sent_tokenize

for i in fomc_df.index:
  fomc_length_df.loc[i,'Sentence_Count']=len(sent_tokenize(fomc_df['Text'][i]))


In [ ]:
fomc_length_df.head()

In [ ]:
## Visulaisation of sentence count distribution of Minutes based on word-count

fomc_length_df[fomc_length_df['Type']=='Minute']['Sentence_Count'].plot(kind='box',figsize=(5,4))
plt.title('Box-plot of distribution of Sentence count of Minutes')
plt.ylabel('Count Frequency')
plt.show()
print('\n Below is the percentile distribution of Minute length based on Sentence count :-\n')
fomc_length_df[fomc_length_df['Type']=='Minute']['Sentence_Count'].describe(percentiles=[0.10, 0.20, 0.50, 0.75, 0.80, 0.9,0.99])


In [ ]:
## Visulaisation of sentence count distribution of Statements based on word-count

fomc_length_df[fomc_length_df['Type']=='Statement']['Sentence_Count'].plot(kind='box',figsize=(5,4))
plt.title('Box-plot of distribution of Sentence count of Statements')
plt.ylabel('Count Frequency')
plt.show()
print('\n Below is the percentile distribution of Statement length based on Sentence count :-\n')
fomc_length_df[fomc_length_df['Type']=='Statement']['Sentence_Count'].describe(percentiles=[0.10, 0.20, 0.50, 0.75, 0.80, 0.9,0.99])


## B] Word Cloud Analysis of FOMC Statements and Meeting Minutes.


In [ ]:
## Pre-processing of text.
import re

fomc_clean_df=pd.DataFrame()
for i in fomc_df.index:
  fomc_clean_df.loc[i,'Cleaned_Text']=fomc_df['Text'][i].lower()  ## Lowercasing text
  fomc_clean_df.loc[i,'Cleaned_Text']=re.sub(r'[^A-Za-z0-9\s]','',fomc_clean_df['Cleaned_Text'][i])  ## REmoving puinctuation
  fomc_clean_df.loc[i,'Cleaned_Text']=re.sub(r'\b\w*\d\w*\b','',fomc_clean_df['Cleaned_Text'][i])  ## Removing numbers
  fomc_clean_df.loc[i,'Cleaned_Text']=re.sub(r'\n+\t+','',fomc_clean_df['Cleaned_Text'][i])  ## Removing empty lines
  fomc_clean_df.loc[i,'Type']=fomc_df['Type'][i]  ## To keep Type column for future references.



In [ ]:
## Lematizing word text post cleaning step

import en_core_web_sm
nlp = en_core_web_sm.load()

for i in fomc_clean_df.index:
  wrd_lst=nlp(fomc_clean_df['Cleaned_Text'][i])
  token=[]
  for wrd in wrd_lst:
    token.append(wrd.lemma_)
    fomc_clean_df.loc[i,'Cleaned_Text']=" ".join(token)

#### Lematization took 12 mins to complete

In [ ]:
## Stopword Removal

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))


for i in fomc_clean_df.index:
  word_tokens=[]
  word_tokens = word_tokenize(fomc_clean_df['Cleaned_Text'][i])
  filtered_text = [word for word in word_tokens if word.lower() not in stop_words]
  fomc_clean_df.loc[i,'Cleaned_Text']=" ".join(filtered_text)

### Word Coud Visualisation of frequently used words in FOMC.

In [ ]:
## Word Cloud visualisation

from collections import Counter
all_txt = ' '.join(fomc_clean_df['Cleaned_Text'])
txt_freq = Counter(all_txt.split())

## Find top 50 most frequency occuring word.
top_50_freq_words = dict(txt_freq.most_common(50))
top_50_freq_words


In [ ]:
# Creating Word Cloud
!pip install wordcloud
from wordcloud import WordCloud
wrdcld = WordCloud(width=800, height=400, background_color='lightyellow', colormap='viridis').generate_from_frequencies(top_50_freq_words)

# Visualize the Word Cloud
plt.figure(figsize=(10, 5))
plt.imshow(wrdcld, interpolation='bilinear')
plt.axis("off")
plt.title("Top 50 Words Using WordCloud \n", fontsize=16)
plt.show()

In [ ]:
## Customising stop words to exclude non-important domain specific words.

custom_stop_words = ['meeting','committee','statement','minute','market','fomc','federal']
stop_words.update(custom_stop_words)

for i in fomc_clean_df.index:
  word_tokens=[]
  word_tokens = word_tokenize(fomc_clean_df['Cleaned_Text'][i])
  filtered_text = [word for word in word_tokens if word.lower() not in stop_words]
  fomc_clean_df.loc[i,'Cleaned_Text']=" ".join(filtered_text)


txt_frq_new = Counter((' '.join(fomc_clean_df['Cleaned_Text'])).split())

wrdcld = WordCloud(width=800, height=400, background_color='lavender', colormap='viridis').generate_from_frequencies(dict(txt_frq_new.most_common(50)))

# Visualize the Word Cloud using custom stopwords
plt.figure(figsize=(10, 5))
plt.imshow(wrdcld, interpolation='bilinear')
plt.axis("off")
plt.title("Top 50 Words Using WordCloud post removal of customised stopwords \n", fontsize=16)
plt.show()

In [ ]:
# Visualize the Word frequncy by word Cloud on Minutes.

wrd_frq_minute=Counter((' '.join(fomc_clean_df[fomc_clean_df['Type']=='Minute']['Cleaned_Text'])).split())
wrdcld = WordCloud(width=800, height=400, background_color='lavender', colormap='magma').generate_from_frequencies(dict(wrd_frq_minute.most_common(50)))

# Visualize the Word Cloud using custom stopwords
plt.figure(figsize=(10, 5))
plt.imshow(wrdcld, interpolation='bilinear')
plt.axis("off")
plt.title("Top 50 Frequently Occuring Words in FOMC Minutes \n", fontsize=16)
plt.show()

In [ ]:
# Visualize the Word frequncy by word Cloud on Statements.

wrd_frq_stmt=Counter((' '.join(fomc_clean_df[fomc_clean_df['Type']=='Statement']['Cleaned_Text'])).split())
wrdcld = WordCloud(width=800, height=400, background_color='lavender', colormap='magma').generate_from_frequencies(dict(wrd_frq_stmt.most_common(50)))

# Visualize the Word Cloud using custom stopwords
plt.figure(figsize=(10, 5))
plt.imshow(wrdcld, interpolation='bilinear')
plt.axis("off")
plt.title("Top 50 Frequently Occuring Words in FOMC Minutes \n", fontsize=16)
plt.show()

### Bi-Gram word pair frequency in FOMC.

In [ ]:
## To create Bi-Gram(pair of words) of preprocessed words for Minutes type.
text_corpus_min=(' '.join(fomc_clean_df[fomc_clean_df['Type']=='Minute']['Cleaned_Text'])).split()  ## Bi-Gram from FOMC minutes
wrd_bigram = [(text_corpus_min[i], text_corpus_min[i + 1]) for i in range(len(text_corpus_min) - 1)]

## To find top 30 bigram frequncy.

wrd_bigram_freq = Counter(wrd_bigram)
wrd_bigram_top_30 = dict(wrd_bigram_freq.most_common(30))

#Print the top 10 words in the bigram frequency

print('Top 15 words in the bigram frequency:-\n')
list(wrd_bigram_top_30)[0:15]


In [ ]:
## Plot of Top 10 Bigram frequency to visualize dsitribution of words pair in FOMC Minutes.

values=list(wrd_bigram_top_30.values())[0:10]    ## Extracting frequency values in list.
keys=[', '.join(val) for val in list(wrd_bigram_top_30.keys())[0:10]]   ## Extracting bi-gram term in a list.

plt.figure(figsize=(6, 4))
plt.bar(keys, values, color='lightcoral')
plt.xticks(rotation=90)
plt.xlabel('Bi-Grams')
plt.ylabel('Frequency')
plt.title('Top ten Bigram frequency in FOMC Minutes')
plt.show()

In [ ]:
## To create Bi-Gram(pair of words) of preprocessed words for Statement type.
text_corpus_stmt=(' '.join(fomc_clean_df[fomc_clean_df['Type']=='Statement']['Cleaned_Text'])).split()  ## Bi-Gram from FOMC Statements
wrd_bigram = [(text_corpus_stmt[i], text_corpus_stmt[i + 1]) for i in range(len(text_corpus_stmt) - 1)]

## To find top 30 bigram frequncy.

wrd_bigram_freq = Counter(wrd_bigram)
wrd_bigram_top_30 = dict(wrd_bigram_freq.most_common(30))

#Print the top 10 words in the bigram frequency

print('Top 15 words in the bigram frequency:-\n')
list(wrd_bigram_top_30)[0:15]


In [ ]:
## Plot of Top 10 Bigram frequency to visualize dsitribution of words pair in FOMC Statements.

values=list(wrd_bigram_top_30.values())[0:10]    ## Extracting frequency values in list.
keys=[', '.join(val) for val in list(wrd_bigram_top_30.keys())[0:10]]   ## Extracting bi-gram term in a list.

plt.figure(figsize=(6, 4))
plt.bar(keys, values, color='lightcoral')
plt.xticks(rotation=90)
plt.xlabel('Bi-Grams')
plt.ylabel('Frequency')
plt.title('Top ten Bigram frequency in FOMC Minutes')
plt.show()

## Topic Moeling omn FOMC data corpus to identify the dominating topics/terms.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

tfidf_vectorize = TfidfVectorizer(stop_words='english', max_df=0.95, min_df=2) ## To extract features from corpus

## max_df and min_df are used to ignore words that appear more than 95% of documents and appear less than 2 in documents.


In [ ]:
## Creating document list.
doc = list((fomc_clean_df[fomc_clean_df['Type']=='Statement']['Cleaned_Text']))

# To create TF-IDF matrix of document.
tfidf = tfidf_vectorize.fit_transform(doc)

## Create dataframe of tfidf matrix.
tfidf_df = pd.DataFrame(tfidf.toarray())

## Extracting feature names.
cols = tfidf_vectorize.get_feature_names_out()
tfidf_df.columns=cols
print(len(doc)) ## Check number of FOMC statements
tfidf_df

### There are 220 statements document and 1048 terms in the statements corpus.

## Topic Modeling by NMF (Non-negative matrix factorization) algorithm.

In [ ]:
from sklearn.decomposition import NMF

## considering 5 topics to identify.
num_topics = 5
nmf_model = NMF(n_components=num_topics, random_state=40)  #write your code here

In [ ]:
##Generating Doc-Topic matrix and Topic-Term matrix using NMF.

W = nmf_model.fit_transform(tfidf)  ## W is Doc-topic matirx
H = nmf_model.components_           ## H is topic-term matirx


print('Rows and Column of W matrix: ',np.shape(W))  ## W matrix is of dimension 21072 X 5
print('Rows and Column of H matrix: ',np.shape(H))  ## H matrix is of dimension 5 X 6767


In [ ]:
##Listing out top 15 terms(based on highest value) of each topic row .

word=np.zeros((5,15)).astype('str')

cols=tfidf_vectorize.get_feature_names_out()

for i in range(0,num_topics):
        word[i]=(cols[np.argsort(H[i])[::-1]][:15])

## Creating dataframe of top 15 words of each topic.

pd.DataFrame(word,index=[ f'Topic{i+1}' for i in range(0,num_topics)] , \
                 columns=[f'Word{i+1}' for i in range(0,15)])

In [ ]:
## Increasing topic_number to 10 for more clarity.

num_topics1 = 10
nmf_model1 = NMF(n_components=num_topics1, random_state=40)

## Decomposing doc-term matrix usng NMF.
W1 = nmf_model1.fit_transform(tfidf)  ## W is Doc-topic matirx
H1 = nmf_model1.components_           ## H is topic-term matirx

print('Rows and Column of W matrix: ',np.shape(W1))  ## W matrix is of dimension 220 X 10
print('Rows and Column of H matrix: ',np.shape(H1))  ## H matrix is of dimension 10 X 1048

##Listing out top 20 words(based on highest value) of each topic row .
word1=np.zeros((10,20)).astype('str')
cols1=tfidf_vectorize.get_feature_names_out()
for i in range(0,num_topics1):
  word1[i]=(cols1[np.argsort(H1[i])[::-1]][:20])

## Creating dataframe of top 20 words of each topic.
pd.DataFrame(word1,index=[ f'Topic{i+1}' for i in range(0,num_topics1)] , \
                 columns=[f'Word{i+1}' for i in range(0,20)])

### Below topics are identified after applying NFM algortihm on FOMC statements.


Topic1 = Inflation, employment and policy direction <br>
Topic2 = FOMC Board participants and procedural decisions <br>
Topic3 = Inflation risk assessment and policy communication  <br>
Topic4 = Assessment of inflation and employment conditions <br>
Topic5 =Inflation and growth<br>
Topic6 = Economic weakness and interest rate reduction <br>
Topic7 = Pandemic-related impact on employment, inflation and policy support<br>
Topic8 = Asset purchases and liquidity and credit support<br>
Topic9 = Monetary policy updates, communication and transparency<br>
Topic10 = Economic recovery, low inflation, and resource utilisation<br>


In [ ]:
# To find max frequency of Topic number in document-term matrix.
W1 = pd.DataFrame(W1, columns=[f'Topic{i + 1}' for i in range(0,num_topics1)]) ## Creating dataframe of W2 matrix.

## Adding new column that assigns Topic Category with highest frequency number.
W1['Max_Topic_Category']=[(W1.iloc[i]).sort_values(ascending=False).index[0] for i in range(0,W1.shape[0])]
W1.value_counts('Max_Topic_Category')  ## To check frequent topic occuring in statememts.


In [ ]:
## Assign topic names in Max_Topic_Catrgory column

topic_map = {
    'Topic1': 'Inflation, employment and policy direction',
    'Topic2': 'FOMC Board participants and procedural decisions',
    'Topic3': 'Inflation risk assessment and policy communication',
    'Topic4': 'Assessment of inflation and employment conditions',
    'Topic5': 'Inflation and growth',
    'Topic6': 'Economic weakness and interest rate reduction',
    'Topic7': 'Pandemic-related impact on employment, inflation and policy support',
    'Topic8': 'Asset purchases and liquidity and credit support',
    'Topic9': 'Monetary policy updates, communication and transparency',
    'Topic10': 'Economic recovery, low inflation, and resource utilisation'
}

##Assigning Category name to each compliant.
W1['Topic'] =W1['Max_Topic_Category'].replace(topic_map)
W1.value_counts('Topic')

In [ ]:
## Verifying the topic name relevance of statements for few documents.

docu_topic_df = pd.DataFrame(fomc_df[fomc_df['Type'] == 'Statement'].Text)
docu_topic_df['Topic_Name'] = W1['Topic'].values

docu_topic_df.head(1)


In [ ]:
docu_topic_df.iloc[[50]]

In [ ]:
docu_topic_df.iloc[[100]]

### Above three samples show that topic names assigned to statements using NMF have correctly assigned the topics.

In [ ]:
## Visualisation of top 5 topic distribution of FOMC statements.
import textwrap
tpc = ['\n'.join(textwrap.wrap(topic, 25)) for topic in W1.value_counts('Topic')[0:5].index.tolist()]
frequency=W1.value_counts('Topic')[0:5].values.tolist()


plt.figure(figsize=(5,4))
plt.bar(tpc, frequency)
plt.xlabel('Topic')
plt.ylabel('Frequency')
plt.title('Topic Distribution of FOMC Statements')
plt.xticks(rotation=90)
plt.show()



In [ ]:
## Performing similar exercise of topic modelling on Minutes.

## Creating document list.
doc = list((fomc_clean_df[fomc_clean_df['Type']=='Minute']['Cleaned_Text']))

# To create TF-IDF matrix of document.
tfidf = tfidf_vectorize.fit_transform(doc)

## Create dataframe of tfidf matrix.
tfidf_df = pd.DataFrame(tfidf.toarray())

## Extracting feature names.
cols = tfidf_vectorize.get_feature_names_out()
tfidf_df.columns=cols
print(len(doc)) ## Check number of FOMC statements
tfidf_df

### There are 238 minutes and 6141 terms in the corpus.

In [ ]:
## Set topic_number to 10.

num_topics1 = 10
nmf_model1 = NMF(n_components=num_topics1, random_state=40)

## Decomposing doc-term matrix usng NMF.
W1 = nmf_model1.fit_transform(tfidf)  ## W is Doc-topic matirx
H1 = nmf_model1.components_           ## H is topic-term matirx

print('Rows and Column of W matrix: ',np.shape(W1))  ## W matrix is of dimension 220 X 10
print('Rows and Column of H matrix: ',np.shape(H1))  ## H matrix is of dimension 10 X 1048

##Listing out top 20 words(based on highest value) of each topic row .
word1=np.zeros((10,20)).astype('str')
cols1=tfidf_vectorize.get_feature_names_out()
for i in range(0,num_topics1):
  word1[i]=(cols1[np.argsort(H1[i])[::-1]][:20])

## Creating dataframe of top 20 words of each topic.
pd.DataFrame(word1,index=[ f'Topic{i+1}' for i in range(0,num_topics1)] , \
                 columns=[f'Word{i+1}' for i in range(0,20)])

### Below topics are identified after applying NFM algortihm on FOMC Minutes.


Topic1 = Assessment of credit, loan conditions and policy stance<br>
Topic2 = Investment and sustainable economic growth<br>
Topic3 = Committee consultations, agreements, and authorisations<br>
Topic4 = Motor vehicle sales and mid-year economic activity<br>
Topic5 = Economic recovery and credit support<br>
Topic6 = Liquidity drop, economy stress and credit conditions<br>
Topic7 = Hurricane-related impacts and third quarter economic activity<br>
Topic8 = Pandemic impacts on credit condition and policy guidance<br>
Topic9 = Geopolitical stress, oil sales, and economic activity<br>
Topic10 = Assessment of GDP and policy normalization<br>


In [ ]:
# To find max frequency of Topic number in document-term matrix.
W1 = pd.DataFrame(W1, columns=[f'Topic{i + 1}' for i in range(0,num_topics1)]) ## Creating dataframe of W2 matrix.

## Adding new column that assigns Topic Category with highest frequency number.
W1['Max_Topic_Category']=[(W1.iloc[i]).sort_values(ascending=False).index[0] for i in range(0,W1.shape[0])]
W1.value_counts('Max_Topic_Category')  ## To check frequent topic occuring in Minutes.


In [ ]:
## Assign topic names in Max_Topic_Catrgory column

topic_map = {
    'Topic1': 'Assessment of credit, loan conditions and policy stance',
    'Topic2': 'Investment and sustainable economic growth',
    'Topic3': 'Committee consultations, agreements, and authorisations',
    'Topic4': 'Motor vehicle sales and mid-year economic activity',
    'Topic5': 'Economic recovery and credit support',
    'Topic6': 'Liquidity drop, economy stress and credit conditions',
    'Topic7': 'Hurricane-related impacts and third quarter economic activity',
    'Topic8': 'Pandemic impacts on credit condition and policy guidance',
    'Topic9': 'Geopolitical stress, oil sales, and economic activity',
    'Topic10': 'Assessment of GDP and policy normalization'
}

##Assigning Category name to each compliant.
W1['Topic'] =W1['Max_Topic_Category'].replace(topic_map)
W1.value_counts('Topic')

In [ ]:
## Visualisation of topic name distribution of FOMC Minute.


tpc = ['\n'.join(textwrap.wrap(topic, 25)) for topic in W1.value_counts('Topic')[0:5].index.tolist()]
frequency=W1.value_counts('Topic')[0:5].values.tolist()


plt.figure(figsize=(5,4))
plt.bar(tpc, frequency)
plt.xlabel('Topic')
plt.ylabel('Frequency')
plt.title('Topic Distribution of FOMC Statements')
plt.xticks(rotation=90)
plt.show()

### Observations from Statements :-<br>
### 1. Documents with a focus on the assessment of the economic market condition based on  inflation and employment rate. <br>

###2. It covers the future policy stance based on the risks and goals communicated by members.  <br>
### 3. Topic related to economic weakness and the activity related to the Reserve Bank growth. <br>
###4. Focus on key macroeconomic indicators, including employment, treasury, inflation, and prices, for future policy guidance.

### Observations from Minutes :-<br>
### 1. nvestment in equipment and software, and the prospect of economic growth, are the key focus of these minutes. <br>

###2. It covers Surveys, GDP assessment, and policy normalization <br>
### 3. Minutes covering credit, loans, delinquency, and future policy amendments related to these topics <br>
###4. Focus on key macroeconomic indicators, including employment, treasury, inflation, and prices, for future policy guidance.<br>
#### 5. Primarily focuses on credit, debt, and loan recovery for a stable economy.  